# 01_connection_test: check the database connection
Writes src/db.py. Host, user and database name are read from .env (git-ignored). The password is always entered in a hidden prompt; it is never written in code, stored in a file, or shown in any output.

## 1. Write the connection helper (src/db.py)

In [1]:
%%writefile ../src/db.py
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL


def get_engine():
    """Engine for the read-only ITSM database.

    Host, user and database name come from .env, which is git-ignored.
    The password is always asked for in a hidden prompt; it is never
    stored in a file or printed.
    """
    load_dotenv(find_dotenv(usecwd=True))
    missing = [k for k in ("DB_HOST", "DB_USER", "DB_NAME") if not os.getenv(k)]
    if missing:
        raise RuntimeError("Set " + ", ".join(missing) + " in .env")
    url = URL.create("mysql+pymysql",
                     username=os.getenv("DB_USER"),
                     password=getpass("Database password: "),
                     host=os.getenv("DB_HOST"),
                     database=os.getenv("DB_NAME"))
    return create_engine(url, pool_pre_ping=True,
                         connect_args={"charset": "utf8mb4", "connect_timeout": 15})

Overwriting ../src/db.py


<details>
<summary><b>Code notes, line by line</b></summary>

- `%%writefile ../src/db.py` : Jupyter magic: saves the rest of the cell as the file src/db.py instead of running it, so every notebook can import the same helper.
- `import os / from getpass import getpass` : os reads environment variables; getpass asks for a password without showing what is typed.
- `from dotenv import load_dotenv, find_dotenv` : python-dotenv reads key=value pairs from the git-ignored .env file into the environment.
- `from sqlalchemy import create_engine / URL` : SQLAlchemy builds the database connection; URL.create assembles the connection string safely (special characters in the password are escaped).
- `def get_engine():` : One function returns a ready connection engine; it takes no arguments, because the password is always asked for.
- `load_dotenv(find_dotenv(usecwd=True))` : Searches upward from the current folder for .env and loads it, so it works from notebooks/.
- `missing = [...]; if missing: raise RuntimeError(...)` : Stops early with a clear message if DB_HOST, DB_USER or DB_NAME is not set (their values are never printed).
- `url = URL.create("mysql+pymysql", ...)` : Builds a MySQL URL that uses the PyMySQL driver, with user, host and database name from .env.
- `password=getpass("Database password: ")` : Asks for the password in a hidden prompt and passes it straight to the connection; it is never stored in a variable or a file.
- `create_engine(url, pool_pre_ping=True, connect_args={...})` : pool_pre_ping tests a connection before use (avoids stale connections); utf8mb4 handles all characters; connect_timeout stops a hang after 15 seconds.

</details>

## 2. Connect and list the tables

In [2]:
import sys, importlib
sys.modules.pop("db", None)          # reload db.py after it is rewritten
importlib.invalidate_caches()
sys.path.append("../src")
from db import get_engine
from sqlalchemy import inspect

engine = get_engine()
print("Tables:", inspect(engine).get_table_names())
engine.dispose()                     # close the connection

Database password:  ········


Tables: ['dataset_list']


<details>
<summary><b>Code notes, line by line</b></summary>

- `import sys, importlib` : sys controls the import path; importlib clears Python's cache of found modules.
- `sys.modules.pop("db", None)` : Forgets any db module imported earlier, so the freshly written db.py is the one loaded.
- `importlib.invalidate_caches()` : Makes Python look at the file system again for new or changed modules.
- `sys.path.append("../src")` : Adds the project's src folder to the import path.
- `from db import get_engine / from sqlalchemy import inspect` : Imports the helper and SQLAlchemy's inspector, which reads database metadata.
- `engine = get_engine()` : Shows the hidden password prompt and creates the engine (no query is run yet).
- `inspect(engine).get_table_names()` : Connects and lists the tables: proves the credentials and network path work.
- `engine.dispose()` : Closes all pooled connections.

</details>